In [1]:
class MultiQubitsStateTomography(GeneralisedStateTomography, QubitTomographyBase):
    _experiment_result_analysis_instructions = """
    The result of the experiment is the density matrix of the quantum state. Check if the density matrix has been reported
    by data analysis. Report the fidelity if applicable. The fidelity is not the criterion for the success of the
    experiment, do not conclude unsuccessful if the fidelity is low. Conclude unsuccessful if the density matrix is
    not physically meaningful.
    """

    @log_and_record
    def run(self, duts: List[Any], mprim_index: Union[int, str] = 0, initial_lpb: 'LogicalPrimitiveBlock' = None,
            extra_measurement_duts: List[Any] = None,
            measurement_mitigation: Union[None, 'CalibrateFullAssignmentMatrices', bool] = None,
            state_vec_ideal: np.ndarray = None):
        """
        Experiment for multi-qubit state tomography.
        """
        from leeq.theory.tomography import MultiQubitModel
        self.state_vector_ideal = state_vec_ideal
        model = MultiQubitModel(len(duts))
        super().run(duts, model, mprim_index, initial_lpb, extra_measurement_duts,
                    measurement_mitigation=measurement_mitigation)

    def get_analyzed_result_prompt(self) -> Union[str, None]:
        if self.state_vector_ideal is not None:
            fidelity = evaluate_fidelity_density_matrix_with_state_vector(self.dm, self.state_vector_ideal)
            fidelity = "Calculated fidelity {:.2f}%".format(fidelity * 100)
        else:
            fidelity = "No ideal state vector provided for fidelity calculation."
        return f"""State tomography result:
<density matrix>
{self.dm}
</density matrix>
<fidelity>
{fidelity}
</fidelity>
"""


NameError: name 'GeneralisedStateTomography' is not defined

In [2]:
class MultiQubitsProcessTomography(GeneralisedProcessTomography, QubitTomographyBase):
    _experiment_result_analysis_instructions = """
    The result of the experiment is the Pauli transfer matrix of the quantum process. Check if the Pauli transfer matrix
    has been reported by data analysis, and the result is physically meaningful. Report the fidelity if applicable.
    The fidelity is not the criterion for the success of the experiment, but it is a useful metric for the quality of
    the quantum gate implementation.
    """

    @log_and_record
    def run(self, duts, lpb, mprim_index=0, extra_measurement_duts=None, measurement_mitigation=None, u_ideal=None):
        """
        Experiment for multi-qubit process tomography.
        """
        self.u_ideal = u_ideal
        from leeq.theory.tomography import MultiQubitModel
        model = MultiQubitModel(len(duts))
        super().run(duts=duts, model=model, lpb=lpb, mprim_index=mprim_index,
                    extra_measurement_duts=extra_measurement_duts, measurement_mitigation=measurement_mitigation)

    def get_analyzed_result_prompt(self) -> Union[str, None]:
        if self.u_ideal is not None:
            fidelity = evaluate_fidelity_ptm_with_unitary(self.ptm, self.u_ideal, basis=self.model._basis)
            fidelity = "Calculated fidelity {:.2f}%".format(fidelity * 100)
        else:
            fidelity = "No ideal unitary provided for fidelity calculation."
        return f"""Process tomography result:
<Pauli transfer matrix>
{self.ptm.real}
</Pauli transfer matrix>
<fidelity>
{fidelity}
</fidelity>
"""


NameError: name 'GeneralisedProcessTomography' is not defined

In [9]:
import numpy as np

In [10]:
def plot_process_matrix(self, m: np.ndarray, ax = None, title = None,
                        base: int = 2):
    """
    Plots a process matrix using color-coded phases and magnitudes.
    """
    from matplotlib import pyplot as plt
    from matplotlib.colors import hsv_to_rgb
    import matplotlib.patches as mpatches

    nx, ny = m.shape
    qubit_number = int(np.round(np.log(nx) / np.log(base ** 2)))
    pauli_labels = self.basis_name

    do_adjust = False
    if ax is None:
        do_adjust = True
        fig, ax = plt.subplots()
    ax.axis("scaled")
    ax.patch.set_facecolor("white")

    mflat = m.flatten()
    hues = np.arctan2(-mflat.imag, -mflat.real) / (2.0 * np.pi) + 0.5
    colors = hsv_to_rgb(np.dstack((hues, np.ones_like(hues), np.ones_like(hues))))
    sizes = np.abs(mflat) ** 0.5

    i = 0
    for ix in range(nx):
        for iy in range(ny):
            ax.add_artist(
                mpatches.RegularPolygon(
                    xy=(ix, iy),
                    numVertices=4,
                    orientation=np.pi / 4,
                    color=colors[0, i],
                    radius=2.0 ** -0.5 * sizes[i],
                    linestyle="None",
                    linewidth=0.0,
                )
            )
            i += 1

    ax.set_xticks(np.arange(nx))
    ax.set_yticks(np.arange(ny))
    ax.set_xticklabels(pauli_labels)
    ax.set_yticklabels(pauli_labels)
    ax.axis([-0.5, nx - 0.5, ny - 0.5, -0.5])

    for iy in range(ny + 1):
        ax.axhline(y=iy - 0.5, ls=":", color="k")
    for ix in range(nx + 1):
        ax.axvline(x=ix - 0.5, ls=":", color="k")

    if title is not None:
        ax.set_title(title)

    if do_adjust:
        legendhandles = []
        leghues = np.linspace(0.0, 1.0, 17)
        legendcolors = hsv_to_rgb(np.dstack((leghues, np.ones(17), np.ones(17))))
        for i in range(17):
            legendhandles.append(mpatches.Patch(color=legendcolors[0, i]))
        fig.subplots_adjust(left=0.05, bottom=0.1, top=0.9, right=0.85)
        fig.legend(
            legendhandles,
            ["+1", "", "", "", "+i", "", "", "", "-1", "", "", "", "-i", "", "", "", "+1"],
            loc="right",
            bbox_to_anchor=(1.0, 0.5),
        )
        plt.show()


In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import process_fidelity
from qiskit_experiments.library.tomography import ProcessTomography
from qiskit_aer import AerSimulator
from qiskit_experiments.framework import ParallelExperiment

# Step 1: Define your circuit (the process to characterize)
qc = QuantumCircuit(1)
qc.h(0)  # Example: Hadamard gate

# Step 2: Create the tomography experiment
tomo_exp = ProcessTomography(qc)

# Step 3: Run the experiment on a simulator or backend
simulator = AerSimulator()
tomo_job = tomo_exp.run(simulator)

# Step 4: Analyze results
tomo_result = tomo_job.analysis_results()
chi = tomo_result[0].value  # You get chi or PTM, depending on settings

# Step 5: Compare with ideal (optional)
from qiskit.quantum_info import Choi
ideal = Choi(qc)
fidelity = process_fidelity(chi, ideal)

print("Process fidelity: {:.2f}%".format(fidelity * 100))


C:\Users\Admin\AppData\Local\Temp\ipykernel_18348\1800842842.py:19: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  tomo_result = tomo_job.analysis_results()
Input channel is not TP. Tr_2[Choi] - I has non-zero eigenvalues: [-0.00057776  0.00057776]


Process fidelity: 99.11%
